In [ ]:

!pip install -q xgboost lightgbm catboost openpyxl scikit-learn pandas numpy matplotlib seaborn


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

uploaded = files.upload()

file_name = next(iter(uploaded))

df = pd.read_excel(file_name)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst five records:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())


Saving Project_datasheet_2019-2020.xlsx to Project_datasheet_2019-2020.xlsx
Dataset shape: (150, 11)

Columns:
['CropType', 'CropDays', 'Soil Moisture', 'Soil Temperature', 'Temperature', 'Humidity', 'Irrigation(Y/N)', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10']

First five records:


,CropType,CropDays,Soil Moisture,Soil Temperature,Temperature,Humidity,Irrigation(Y/N),Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
0,1,1,630,19,33,52,1,NaN,NaN,NaN,NaN
1,1,2,460,20,31,50,0,NaN,NaN,NaN,NaN
2,1,3,230,22,33,48,0,NaN,NaN,CropType,Code
3,1,4,140,24,28,62,0,NaN,NaN,Paddy,1
4,1,5,720,18,28,63,1,NaN,NaN,Ground Nuts,2



Missing values:
CropType              0
CropDays              0
Soil Moisture         0
Soil Temperature      0
Temperature           0
Humidity              0
Irrigation(Y/N)       0
Unnamed: 7          150
Unnamed: 8          150
Unnamed: 9          144
Unnamed: 10         144
dtype: int64


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Remove duplicate records
df = df.drop_duplicates().copy()

# Clean column names
df.columns = df.columns.str.strip()

# Target column
target = "Irrigation(Y/N)"

# Separate input and target
X = df.drop(columns=[target])
y = df[target]

# Handle missing values
for col in X.columns:
    if X[col].dtype == "object":
        X[col] = X[col].fillna(X[col].mode()[0])
    else:
        X[col] = X[col].fillna(X[col].median())

# Encode categorical features
X = pd.get_dummies(X, drop_first=False, dtype=int)

# Encode target
if y.dtype == "object":
    y = y.astype(str).str.strip().str.upper()
    y = y.replace({
        "Y": 1, "YES": 1, "N": 0, "NO": 0
    })

y = pd.to_numeric(y, errors="raise").astype(int)

print("Processed features:", X.columns.tolist())
print("\nTarget distribution:")
print(y.value_counts())

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))


Processed features: ['CropType', 'CropDays', 'Soil Moisture', 'Soil Temperature', 'Temperature', 'Humidity', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9_CropType', 'Unnamed: 9_Ground Nuts', 'Unnamed: 9_Irrigation', 'Unnamed: 9_No (N)', 'Unnamed: 9_Paddy', 'Unnamed: 9_Yes (Y)', 'Unnamed: 10_0', 'Unnamed: 10_1', 'Unnamed: 10_2', 'Unnamed: 10_Code']

Target distribution:
Irrigation(Y/N)
0    125
1     25
Name: count, dtype: int64

Training records: 120
Testing records: 30


/tmp/ipykernel_1824/58092300.py:20: UserWarning: Unable to sort modes: '<' not supported between instances of 'int' and 'str'
  X[col] = X[col].fillna(X[col].mode()[0])


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Remove duplicate records
df = df.drop_duplicates().copy()

# Clean column names
df.columns = df.columns.str.strip()

# Target column
target = "Irrigation(Y/N)"

# Separate input and target
X = df.drop(columns=[target])
y = df[target]

# Handle missing values
for col in X.columns:
    if X[col].dtype == "object":
        X[col] = X[col].fillna(X[col].mode()[0])
    else:
        X[col] = X[col].fillna(X[col].median())

# Encode categorical features
X = pd.get_dummies(X, drop_first=False, dtype=int)

# Encode target
if y.dtype == "object":
    y = y.astype(str).str.strip().str.upper()
    y = y.replace({
        "Y": 1, "YES": 1, "N": 0, "NO": 0
    })

y = pd.to_numeric(y, errors="raise").astype(int)

print("Processed features:", X.columns.tolist())
print("\nTarget distribution:")
print(y.value_counts())

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))


Processed features: ['CropType', 'CropDays', 'Soil Moisture', 'Soil Temperature', 'Temperature', 'Humidity', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9_CropType', 'Unnamed: 9_Ground Nuts', 'Unnamed: 9_Irrigation', 'Unnamed: 9_No (N)', 'Unnamed: 9_Paddy', 'Unnamed: 9_Yes (Y)', 'Unnamed: 10_0', 'Unnamed: 10_1', 'Unnamed: 10_2', 'Unnamed: 10_Code']

Target distribution:
Irrigation(Y/N)
0    125
1     25
Name: count, dtype: int64

Training records: 120
Testing records: 30


/tmp/ipykernel_1824/58092300.py:20: UserWarning: Unable to sort modes: '<' not supported between instances of 'int' and 'str'
  X[col] = X[col].fillna(X[col].mode()[0])


In [ ]:

from sklearn.ensemble import RandomForestClassifier

rf_selector = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

rf_selector.fit(X_train, y_train)

rf_importance = pd.Series(
    rf_selector.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("Random Forest Feature Importance:")
print(rf_importance)

selected_rf = rf_importance[
    rf_importance > 0
].index.tolist()

print("\nSelected features:", selected_rf)


Random Forest Feature Importance:
Soil Moisture             0.458754
Soil Temperature          0.288175
CropDays                  0.095477
Humidity                  0.067830
Temperature               0.064207
CropType                  0.014697
Unnamed: 10_2             0.002896
Unnamed: 9_CropType       0.002025
Unnamed: 9_Ground Nuts    0.001928
Unnamed: 10_Code          0.001596
Unnamed: 9_Irrigation     0.000615
Unnamed: 10_1             0.000578
Unnamed: 9_No (N)         0.000541
Unnamed: 10_0             0.000482
Unnamed: 9_Yes (Y)        0.000198
Unnamed: 8                0.000000
Unnamed: 7                0.000000
Unnamed: 9_Paddy          0.000000
dtype: float64

Selected features: ['Soil Moisture', 'Soil Temperature', 'CropDays', 'Humidity', 'Temperature', 'CropType', 'Unnamed: 10_2', 'Unnamed: 9_CropType', 'Unnamed: 9_Ground Nuts', 'Unnamed: 10_Code', 'Unnamed: 9_Irrigation', 'Unnamed: 10_1', 'Unnamed: 9_No (N)', 'Unnamed: 10_0', 'Unnamed: 9_Yes (Y)']


In [ ]:

from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

rfe_estimator = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rfe = RFE(
    estimator=rfe_estimator,
    n_features_to_select=max(1, X_train.shape[1] // 2)
)

rfe.fit(X_train, y_train)

selected_rfe = X_train.columns[
    rfe.support_
].tolist()

print("RFE Selected Features:")
print(selected_rfe)


RFE Selected Features:
['CropType', 'CropDays', 'Soil Moisture', 'Soil Temperature', 'Temperature', 'Humidity', 'Unnamed: 9_Ground Nuts', 'Unnamed: 10_2', 'Unnamed: 10_Code']


In [ ]:

print("Missing values in training data:")
print(X_train.isnull().sum())

print("\nTotal missing values:")
print(X_train.isnull().sum().sum())


Missing values in training data:
CropType                    0
CropDays                    0
Soil Moisture               0
Soil Temperature            0
Temperature                 0
Humidity                    0
Unnamed: 7                120
Unnamed: 8                120
Unnamed: 9_CropType         0
Unnamed: 9_Ground Nuts      0
Unnamed: 9_Irrigation       0
Unnamed: 9_No (N)           0
Unnamed: 9_Paddy            0
Unnamed: 9_Yes (Y)          0
Unnamed: 10_0               0
Unnamed: 10_1               0
Unnamed: 10_2               0
Unnamed: 10_Code            0
dtype: int64

Total missing values:
240


In [ ]:
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd

# Replace infinite values with NaN
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

# Keep all columns, even those containing only NaN
imputer = SimpleImputer(strategy="median", keep_empty_features=True)

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Remaining NaN values:", X_train.isna().sum().sum())

X_train shape: (120, 18)
X_test shape: (30, 18)
Remaining NaN values: 0


In [ ]:

from sklearn.feature_selection import SelectKBest, mutual_info_classif

k = max(1, X_train.shape[1] // 2)

skb = SelectKBest(
    score_func=mutual_info_classif,
    k=k
)

skb.fit(X_train, y_train)

selected_skb = X_train.columns[
    skb.get_support()
].tolist()

print("SelectKBest Selected Features:")
print(selected_skb)

print("\nMutual Information Scores:")
print(pd.Series(
    skb.scores_,
    index=X_train.columns
).sort_values(ascending=False))


SelectKBest Selected Features:
['Soil Moisture', 'Soil Temperature', 'Unnamed: 9_No (N)', 'Unnamed: 9_Paddy', 'Unnamed: 9_Yes (Y)', 'Unnamed: 10_0', 'Unnamed: 10_1', 'Unnamed: 10_2', 'Unnamed: 10_Code']

Mutual Information Scores:
Soil Moisture             0.304951
Soil Temperature          0.300736
Unnamed: 9_Yes (Y)        0.061572
Unnamed: 9_No (N)         0.024768
Unnamed: 9_Paddy          0.002779
CropType                  0.000000
CropDays                  0.000000
Unnamed: 7                0.000000
Temperature               0.000000
Humidity                  0.000000
Unnamed: 9_Ground Nuts    0.000000
Unnamed: 9_CropType       0.000000
Unnamed: 8                0.000000
Unnamed: 9_Irrigation     0.000000
Unnamed: 10_0             0.000000
Unnamed: 10_1             0.000000
Unnamed: 10_2             0.000000
Unnamed: 10_Code          0.000000
dtype: float64


In [ ]:
# Rank features from all three methods

rf_rank = rf_importance.rank(ascending=False, pct=True)

rfe_rank = pd.Series(
    rfe.ranking_,
    index=X_train.columns
).rank(ascending=True, pct=True)

mi_scores = pd.Series(
    skb.scores_,
    index=X_train.columns
)

mi_rank = mi_scores.rank(ascending=False, pct=True)

# Calculate combined hybrid score
hybrid_score = (rf_rank + rfe_rank + mi_rank) / 3

# Sort features by hybrid score
hybrid_ranking = hybrid_score.sort_values(ascending=True)

# Select the top 50% of features
n_hybrid = max(1, X_train.shape[1] // 2)

selected_hybrid = hybrid_ranking.head(n_hybrid).index.tolist()

print("Hybrid selected features:")
print(selected_hybrid)

print("Number of selected features:", len(selected_hybrid))

Hybrid selected features:
['Soil Moisture', 'Soil Temperature', 'CropDays', 'Humidity', 'Temperature', 'CropType', 'Unnamed: 10_2', 'Unnamed: 9_Ground Nuts', 'Unnamed: 10_Code']
Number of selected features: 9


In [ ]:
!pip install -q xgboost lightgbm catboost

In [ ]:
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

In [ ]:
models = {
    "AdaBoost": AdaBoostClassifier(
        n_estimators=100,
        learning_rate=0.5,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=2,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=2,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        verbosity=-1,
        random_state=42
    ),

    "CatBoost": CatBoostClassifier(
        iterations=100,
        depth=3,
        learning_rate=0.05,
        verbose=0,
        random_seed=42,
        allow_writing_files=False
    )
}

In [ ]:
feature_methods = {
    "Original Features": X_train.columns.tolist(),
    "Random Forest": selected_rf,
    "RFE": selected_rfe,
    "SelectKBest": selected_skb,
    "Hybrid": selected_hybrid
}

In [ ]:
import re

# Create a mapping for clean feature names
clean_columns = {
    col: re.sub(r'[^A-Za-z0-9_]+', '_', str(col)).strip('_')
    for col in X_train.columns
}

# Rename training and testing columns consistently
X_train = X_train.rename(columns=clean_columns)
X_test = X_test.rename(columns=clean_columns)

# Update selected feature lists
selected_rf = [clean_columns[col] for col in selected_rf]
selected_rfe = [clean_columns[col] for col in selected_rfe]
selected_skb = [clean_columns[col] for col in selected_skb]
selected_hybrid = [clean_columns[col] for col in selected_hybrid]

print("Cleaned feature names:")
print(X_train.columns.tolist())

Cleaned feature names:
['CropType', 'CropDays', 'Soil_Moisture', 'Soil_Temperature', 'Temperature', 'Humidity', 'Unnamed_7', 'Unnamed_8', 'Unnamed_9_CropType', 'Unnamed_9_Ground_Nuts', 'Unnamed_9_Irrigation', 'Unnamed_9_No_N', 'Unnamed_9_Paddy', 'Unnamed_9_Yes_Y', 'Unnamed_10_0', 'Unnamed_10_1', 'Unnamed_10_2', 'Unnamed_10_Code']


In [ ]:
feature_methods = {
    "Original Features": X_train.columns.tolist(),
    "Random Forest": selected_rf,
    "RFE": selected_rfe,
    "SelectKBest": selected_skb,
    "Hybrid": selected_hybrid
}

In [ ]:
results = []
confusion_matrices = {}

for method_name, selected_features in feature_methods.items():

    # Use the selected features
    X_train_selected = X_train[selected_features]
    X_test_selected = X_test[selected_features]

    for model_name, model in models.items():

        # Create a fresh model for each experiment
        clf = clone(model)

        # Train
        clf.fit(X_train_selected, y_train)

        # Predict
        y_pred = clf.predict(X_test_selected)

        # Prediction probabilities for ROC-AUC
        y_prob = clf.predict_proba(X_test_selected)[:, 1]

        # Calculate evaluation metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(
            y_test, y_pred, zero_division=0
        )
        recall = recall_score(
            y_test, y_pred, zero_division=0
        )
        f1 = f1_score(
            y_test, y_pred, zero_division=0
        )

        try:
            auc = roc_auc_score(y_test, y_prob)
        except ValueError:
            auc = np.nan

        results.append({
            "Feature Selection": method_name,
            "Boosting Algorithm": model_name,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1,
            "ROC-AUC": auc
        })

        confusion_matrices[(method_name, model_name)] = (
            confusion_matrix(y_test, y_pred)
        )

# Create the final results table
results_df = pd.DataFrame(results)

# Display results as percentages
metric_columns = [
    "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"
]

results_df[metric_columns] = (
    results_df[metric_columns] * 100
).round(2)

print("Total experiments:", len(results_df))
display(results_df)

Total experiments: 25


,Feature Selection,Boosting Algorithm,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Original Features,AdaBoost,96.67,100.0,80.0,88.89,100.0
1,Original Features,Gradient Boosting,96.67,100.0,80.0,88.89,100.0
2,Original Features,XGBoost,96.67,100.0,80.0,88.89,99.2
3,Original Features,LightGBM,96.67,100.0,80.0,88.89,100.0
4,Original Features,CatBoost,96.67,100.0,80.0,88.89,99.2
5,Random Forest,AdaBoost,96.67,100.0,80.0,88.89,100.0
6,Random Forest,Gradient Boosting,96.67,100.0,80.0,88.89,100.0
7,Random Forest,XGBoost,96.67,100.0,80.0,88.89,99.2
8,Random Forest,LightGBM,96.67,100.0,80.0,88.89,100.0
9,Random Forest,CatBoost,96.67,100.0,80.0,88.89,99.2


In [ ]:
print("Total experiments:", len(results_df))

display(
    results_df.sort_values(
        by="Accuracy",
        ascending=False
    )
)

Total experiments: 25


,Feature Selection,Boosting Algorithm,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Original Features,AdaBoost,96.67,100.0,80.0,88.89,100.0
1,Original Features,Gradient Boosting,96.67,100.0,80.0,88.89,100.0
2,Original Features,XGBoost,96.67,100.0,80.0,88.89,99.2
3,Original Features,LightGBM,96.67,100.0,80.0,88.89,100.0
4,Original Features,CatBoost,96.67,100.0,80.0,88.89,99.2
5,Random Forest,AdaBoost,96.67,100.0,80.0,88.89,100.0
6,Random Forest,Gradient Boosting,96.67,100.0,80.0,88.89,100.0
7,Random Forest,XGBoost,96.67,100.0,80.0,88.89,99.2
8,Random Forest,LightGBM,96.67,100.0,80.0,88.89,100.0
9,Random Forest,CatBoost,96.67,100.0,80.0,88.89,99.2


In [ ]:
metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "ROC-AUC"
]

comparison = results_df.groupby(
    "Feature Selection"
)[metric_columns].mean().round(2)

print("Average performance of each feature-selection method:")
display(comparison)

Average performance of each feature-selection method:


,Accuracy,Precision,Recall,F1-Score,ROC-AUC
Feature Selection,,,,,
Hybrid,96.67,100.0,80.0,88.89,99.68
Original Features,96.67,100.0,80.0,88.89,99.68
RFE,96.67,100.0,80.0,88.89,99.68
Random Forest,96.67,100.0,80.0,88.89,99.68
SelectKBest,96.67,100.0,80.0,88.89,98.72


In [ ]:
print("Selected feature counts:")

for name, features in feature_methods.items():
    print(name, ":", len(features), features)

print("\nNumber of experiments:", len(results_df))

print("\nResults for each algorithm:")
display(
    results_df.sort_values(
        ["Feature Selection", "Boosting Algorithm"]
    )
)

Selected feature counts:
Original Features : 18 ['CropType', 'CropDays', 'Soil_Moisture', 'Soil_Temperature', 'Temperature', 'Humidity', 'Unnamed_7', 'Unnamed_8', 'Unnamed_9_CropType', 'Unnamed_9_Ground_Nuts', 'Unnamed_9_Irrigation', 'Unnamed_9_No_N', 'Unnamed_9_Paddy', 'Unnamed_9_Yes_Y', 'Unnamed_10_0', 'Unnamed_10_1', 'Unnamed_10_2', 'Unnamed_10_Code']
Random Forest : 15 ['Soil_Moisture', 'Soil_Temperature', 'CropDays', 'Humidity', 'Temperature', 'CropType', 'Unnamed_10_2', 'Unnamed_9_CropType', 'Unnamed_9_Ground_Nuts', 'Unnamed_10_Code', 'Unnamed_9_Irrigation', 'Unnamed_10_1', 'Unnamed_9_No_N', 'Unnamed_10_0', 'Unnamed_9_Yes_Y']
RFE : 9 ['CropType', 'CropDays', 'Soil_Moisture', 'Soil_Temperature', 'Temperature', 'Humidity', 'Unnamed_9_Ground_Nuts', 'Unnamed_10_2', 'Unnamed_10_Code']
SelectKBest : 9 ['Soil_Moisture', 'Soil_Temperature', 'Unnamed_9_No_N', 'Unnamed_9_Paddy', 'Unnamed_9_Yes_Y', 'Unnamed_10_0', 'Unnamed_10_1', 'Unnamed_10_2', 'Unnamed_10_Code']
Hybrid : 9 ['Soil_Moisture

,Feature Selection,Boosting Algorithm,Accuracy,Precision,Recall,F1-Score,ROC-AUC
20,Hybrid,AdaBoost,96.67,100.0,80.0,88.89,100.0
24,Hybrid,CatBoost,96.67,100.0,80.0,88.89,99.2
21,Hybrid,Gradient Boosting,96.67,100.0,80.0,88.89,100.0
23,Hybrid,LightGBM,96.67,100.0,80.0,88.89,100.0
22,Hybrid,XGBoost,96.67,100.0,80.0,88.89,99.2
0,Original Features,AdaBoost,96.67,100.0,80.0,88.89,100.0
4,Original Features,CatBoost,96.67,100.0,80.0,88.89,99.2
1,Original Features,Gradient Boosting,96.67,100.0,80.0,88.89,100.0
3,Original Features,LightGBM,96.67,100.0,80.0,88.89,100.0
2,Original Features,XGBoost,96.67,100.0,80.0,88.89,99.2


In [ ]:
print("Total experiments:", len(results_df))

print("\nExperiments per feature-selection method:")
print(results_df.groupby("Feature Selection").size())

print("\nExperiments per boosting algorithm:")
print(results_df.groupby("Boosting Algorithm").size())

print("\nDuplicate experiment combinations:")
print(
    results_df.duplicated(
        subset=["Feature Selection", "Boosting Algorithm"]
    ).sum()
)

print("\nSelected features:")
for method, features in feature_methods.items():
    print(f"{method}: {len(features)} features")
    print(features)

Total experiments: 25

Experiments per feature-selection method:
Feature Selection
Hybrid               5
Original Features    5
RFE                  5
Random Forest        5
SelectKBest          5
dtype: int64

Experiments per boosting algorithm:
Boosting Algorithm
AdaBoost             5
CatBoost             5
Gradient Boosting    5
LightGBM             5
XGBoost              5
dtype: int64

Duplicate experiment combinations:
0

Selected features:
Original Features: 18 features
['CropType', 'CropDays', 'Soil_Moisture', 'Soil_Temperature', 'Temperature', 'Humidity', 'Unnamed_7', 'Unnamed_8', 'Unnamed_9_CropType', 'Unnamed_9_Ground_Nuts', 'Unnamed_9_Irrigation', 'Unnamed_9_No_N', 'Unnamed_9_Paddy', 'Unnamed_9_Yes_Y', 'Unnamed_10_0', 'Unnamed_10_1', 'Unnamed_10_2', 'Unnamed_10_Code']
Random Forest: 15 features
['Soil_Moisture', 'Soil_Temperature', 'CropDays', 'Humidity', 'Temperature', 'CropType', 'Unnamed_10_2', 'Unnamed_9_CropType', 'Unnamed_9_Ground_Nuts', 'Unnamed_10_Code', 'Unnamed_

In [ ]:
print("Training target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

print("\nTraining class percentages:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting class percentages:")
print(y_test.value_counts(normalize=True) * 100)

Training target distribution:
Irrigation(Y/N)
0    100
1     20
Name: count, dtype: int64

Testing target distribution:
Irrigation(Y/N)
0    25
1     5
Name: count, dtype: int64

Training class percentages:
Irrigation(Y/N)
0    83.333333
1    16.666667
Name: proportion, dtype: float64

Testing class percentages:
Irrigation(Y/N)
0    83.333333
1    16.666667
Name: proportion, dtype: float64


In [ ]:
for key, matrix in confusion_matrices.items():
    print("\nFeature Selection:", key[0])
    print("Boosting Algorithm:", key[1])
    print(matrix)


Feature Selection: Original Features
Boosting Algorithm: AdaBoost
[[25  0]
 [ 1  4]]

Feature Selection: Original Features
Boosting Algorithm: Gradient Boosting
[[25  0]
 [ 1  4]]

Feature Selection: Original Features
Boosting Algorithm: XGBoost
[[25  0]
 [ 1  4]]

Feature Selection: Original Features
Boosting Algorithm: LightGBM
[[25  0]
 [ 1  4]]

Feature Selection: Original Features
Boosting Algorithm: CatBoost
[[25  0]
 [ 1  4]]

Feature Selection: Random Forest
Boosting Algorithm: AdaBoost
[[25  0]
 [ 1  4]]

Feature Selection: Random Forest
Boosting Algorithm: Gradient Boosting
[[25  0]
 [ 1  4]]

Feature Selection: Random Forest
Boosting Algorithm: XGBoost
[[25  0]
 [ 1  4]]

Feature Selection: Random Forest
Boosting Algorithm: LightGBM
[[25  0]
 [ 1  4]]

Feature Selection: Random Forest
Boosting Algorithm: CatBoost
[[25  0]
 [ 1  4]]

Feature Selection: RFE
Boosting Algorithm: AdaBoost
[[25  0]
 [ 1  4]]

Feature Selection: RFE
Boosting Algorithm: Gradient Boosting
[[25  0]
 

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE, SelectKBest, mutual_info_classif
from scipy.stats import rankdata
import numpy as np

class HybridSelector(BaseEstimator, TransformerMixin):

    def fit(self, X, y):
        X = np.asarray(X)
        n_features = X.shape[1]
        k = max(1, n_features // 2)

        rf = RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight="balanced"
        )
        rf.fit(X, y)

        rfe = RFE(
            estimator=RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                class_weight="balanced"
            ),
            n_features_to_select=k
        )
        rfe.fit(X, y)

        mi = mutual_info_classif(X, y, random_state=42)

        rf_rank = rankdata(-rf.feature_importances_) / n_features
        rfe_rank = rankdata(rfe.ranking_) / n_features
        mi_rank = rankdata(-mi) / n_features

        hybrid_score = (rf_rank + rfe_rank + mi_rank) / 3

        self.selected_indices_ = np.argsort(hybrid_score)[:k]

        return self

    def transform(self, X):
        return np.asarray(X)[:, self.selected_indices_]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import StratifiedKFold, cross_validate

selectors = {
    "Original Features": "passthrough",

    "Random Forest": SelectFromModel(
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight="balanced"
        ),
        threshold="mean"
    ),

    "RFE": RFE(
        estimator=RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight="balanced"
        ),
        n_features_to_select=max(1, X_train.shape[1] // 2)
    ),

    "SelectKBest": SelectKBest(
        score_func=mutual_info_classif,
        k=max(1, X_train.shape[1] // 2)
    ),

    "Hybrid": HybridSelector()
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = []

for method_name, selector in selectors.items():

    for model_name, model in models.items():

        pipeline = Pipeline([
            ("feature_selection", clone(selector)
             if selector != "passthrough" else "passthrough"),
            ("classifier", clone(model))
        ])

        scores = cross_validate(
            pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=1
        )

        cv_results.append({
            "Feature Selection": method_name,
            "Boosting Algorithm": model_name,
            "Accuracy Mean": scores["test_accuracy"].mean() * 100,
            "Accuracy Std": scores["test_accuracy"].std() * 100,
            "Precision Mean": scores["test_precision"].mean() * 100,
            "Recall Mean": scores["test_recall"].mean() * 100,
            "F1 Mean": scores["test_f1"].mean() * 100,
            "ROC-AUC Mean": scores["test_roc_auc"].mean() * 100
        })

cv_results_df = pd.DataFrame(cv_results)

display(cv_results_df.round(2))

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

,Feature Selection,Boosting Algorithm,Accuracy Mean,Accuracy Std,Precision Mean,Recall Mean,F1 Mean,ROC-AUC Mean
0,Original Features,AdaBoost,93.33,4.25,86.67,75.0,77.90,97.62
1,Original Features,Gradient Boosting,90.83,1.67,76.67,70.0,70.76,93.00
2,Original Features,XGBoost,95.00,1.67,93.33,80.0,84.57,95.38
3,Original Features,LightGBM,94.17,2.04,86.67,85.0,83.43,95.00
4,Original Features,CatBoost,94.17,2.04,88.33,80.0,82.43,97.25
5,Random Forest,AdaBoost,93.33,4.25,86.67,75.0,77.90,96.62
6,Random Forest,Gradient Boosting,90.83,1.67,76.67,70.0,70.76,93.00
7,Random Forest,XGBoost,95.00,1.67,93.33,80.0,84.57,95.38
8,Random Forest,LightGBM,94.17,2.04,86.67,85.0,83.43,94.88
9,Random Forest,CatBoost,94.17,2.04,88.33,80.0,82.43,98.25


In [ ]:
cv_comparison = cv_results_df.groupby(
    "Feature Selection"
)[[
    "Accuracy Mean",
    "Precision Mean",
    "Recall Mean",
    "F1 Mean",
    "ROC-AUC Mean"
]].mean().round(2)

display(cv_comparison)

,Accuracy Mean,Precision Mean,Recall Mean,F1 Mean,ROC-AUC Mean
Feature Selection,,,,,
Hybrid,93.67,86.33,79.0,80.39,95.50
Original Features,93.50,86.33,78.0,79.82,95.65
RFE,93.67,86.33,79.0,80.39,95.72
Random Forest,93.50,86.33,78.0,79.82,95.62
SelectKBest,91.83,81.67,71.0,72.75,95.18


In [ ]:
best = results_df.loc[results_df["Accuracy"].idxmax()]

print("BEST COMBINATION")
print("-------------------------")
print("Boosting Algorithm:", best["Boosting Algorithm"])
print("Feature Selection:", best["Feature Selection"])
print("Highest Accuracy:", best["Accuracy"], "%")

BEST COMBINATION
-------------------------
Boosting Algorithm: AdaBoost
Feature Selection: Original Features
Highest Accuracy: 96.67 %
